In [11]:
import pygame
import random

# --- Setup ---
pygame.init()

WIDTH, HEIGHT = 1280, 720
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Catch the squares")
clock = pygame.time.Clock()
font = pygame.font.Font(None, 36)

# Colors (R, G, B)
MIDNIGHT = (39, 51, 94)
BLACK = (0, 0, 0)
RAINBOW = (random.randint(0,255),random.randint(0,255),random.randint(0,255))
WHITE = (255, 255, 255)
RED = (255, 0, 0)

# --- Player ---
player_x = WIDTH // 2
player_y = HEIGHT // 2
player_size = 40
player_speed = 5

# --- Target ---
target_size = 50
target_x = random.randint(0, WIDTH - target_size)
target_y = random.randint(0, HEIGHT - target_size)
SCORE = 0

# --- 2nd Target ---
target2_size = 30
target2_x = random.randint(0, WIDTH - target2_size)
target2_y = random.randint(0, HEIGHT - target2_size)
target2_xd = random.randint(0,1)
target2_yd = random.randint(0,1)



# --- Play button (used in menu state) ---
play_button = pygame.Rect(WIDTH // 2 - 60, HEIGHT // 2 - 25, 120, 50)

# --- Game state ---
# "menu" = showing start screen, "playing" = game running
game_state = "menu"

# --- Game loop ---
running = True
while running:

    # 1. Handle events
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.MOUSEBUTTONDOWN and (game_state == "menu" or  game_state == "pause"):
            if play_button.collidepoint(event.pos):
                game_state = "playing"
                SCORE = 0
                start_timer = pygame.time.get_ticks()

    

    # 2. Update + draw, depending on state
    if game_state == "menu":
        screen.fill(RAINBOW)
        
        pygame.draw.rect(screen, WHITE, play_button)
        play_text = font.render("PLAY", True, BLACK)
        screen.blit(play_text, (play_button.x + 25, play_button.y + 12))


    elif game_state == "pause":
        screen.fill(MIDNIGHT)
        result_text = font.render(f"Score: {SCORE} points!", True, WHITE)
        screen.blit(result_text, (WIDTH//2 - 150, HEIGHT//2 - 100))
        pygame.draw.rect(screen, WHITE, play_button)
        retry_text = font.render("RETRY", True, BLACK)
        screen.blit(retry_text, (play_button.x + 25, play_button.y + 12))

    elif game_state == "playing":

        # --- Update ---
        e_seconds = (pygame.time.get_ticks() - start_timer)
        keys = pygame.key.get_pressed()

        if keys[pygame.K_LSHIFT]:
            boost = 2
        else:
            boost = 1


        if keys[pygame.K_LEFT]:
            player_x -= player_speed * boost
        if keys[pygame.K_RIGHT]:
            player_x += player_speed * boost
        if keys[pygame.K_UP]:
            player_y -= player_speed * boost
        if keys[pygame.K_DOWN]:
            player_y += player_speed * boost
        
        # Update 2nd target

        if target2_xd == 0:    #if 0, target2 go left, else go right
            target2_x -= 6
        else:
            target2_x += 6

        if target2_x < 0:     #if target2 reaches left side, change direction
            target2_xd = 1
        elif target2_x > WIDTH:
            target2_xd = 0
        else:
            pass
            
        if target2_yd == 0:    #if 0, target2 go left, else go right
            target2_y -= 6
        else:
            target2_y += 6

        if target2_y < 0:     #if target2 reaches left side, change direction
            target2_yd = 1
        elif target2_y > HEIGHT:
            target2_yd = 0
        else:
            pass
            

        # Quit game
        if e_seconds > 10000:
            game_state = "pause"

        if keys[pygame.K_ESCAPE]:
            running = False

        # TASK: Keep player inside the border
        if player_x < 40:
            player_x = 40
        if player_x > WIDTH - 40:
            player_x = WIDTH - 40

        if player_y < 40:
            player_y = 40
        if player_y > HEIGHT - 40:
            player_y = HEIGHT - 40



        # --- Draw ---
        screen.fill(MIDNIGHT)
        
        #pygame.draw.rect(screen, WHITE, (player_x, player_y, player_size, player_size))
        pygame.draw.circle(screen, WHITE, (player_x, player_y), 40)
        pygame.draw.rect(screen, RED, (target_x, target_y, target_size, target_size))
        pygame.draw.rect(screen, RAINBOW, (target2_x, target2_y, target2_size, target2_size))


        #draw hitboxes
        player_rect = pygame.Rect(player_x - 40, player_y - 40, 80, 80)
        target_rect = pygame.Rect(target_x, target_y, target_size, target_size)
        target2_rect = pygame.Rect(target2_x, target2_y, target2_size, target2_size)
 
        if player_rect.colliderect(target_rect):
            target_x = random.randint(0, WIDTH - target_size)
            target_y = random.randint(0, HEIGHT - target_size)
            pygame.draw.rect(screen, RED, (target_x, target_y, target_size, target_size))
            SCORE = SCORE + 1

        if player_rect.colliderect(target2_rect):
            target2_x = random.randint(0, WIDTH - target2_size)
            target2_y = random.randint(0, HEIGHT - target2_size)
            pygame.draw.rect(screen, RED, (target2_x, target2_y, target2_size, target2_size))
            SCORE = SCORE + 2
            target2_xd = random.randint(0,1)
            target2_yd = random.randint(0,1)

        # TASK: Draw UI (score, timer, instructions)
        score_text = font.render(f"Score: {SCORE} pts", True, WHITE)
        screen.blit(score_text, (50, 50))

        timer_text = font.render(f"Time: {round((10 - e_seconds/1000), 2)} s", True, WHITE)
        screen.blit(timer_text, (50, 100))
        #screen.blit(play_text, (play_button.x + 25, play_button.y + 12))

    # 3. Show what we drew
    pygame.display.flip()

    # 4. Control frame rate
    clock.tick(60)

pygame.quit()